# Домашнее задание 2
Краткий анализ датасета Titanic.

## Загрузка данных
Считываем таблицу из файла `train.csv`.

In [ ]:
import re
import pandas as pd

df = pd.read_csv('train.csv')
df.head()


## Основная информация
Смотрим типы данных, пропуски и основные статистики.

In [ ]:
print(f'Размер датасета: {df.shape[0]} строк и {df.shape[1]} столбцов')
print()
df.info()
print()
print('Пропуски по столбцам:')
display(df.isna().sum())
print('Средние и другие статистики по числовым столбцам:')
display(df.describe())
print('Статистики по категориальным столбцам:')
display(df.describe(include='object'))


## Выживаемость по классам
Считаем долю выживших в процентах для каждого `Pclass`.

In [ ]:
survival_by_class = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)
survival_by_class


## Подготовка имен
Выделяем имя пассажира из поля `Name`.

In [ ]:
def extract_first_name(name: str, sex: str) -> str:
    match = re.search(r'\(([^)]+)\)', name)
    if sex == 'female' and match:
        return match.group(1).strip().split()[0].strip('"\'.,')
    part = name.split(',', 1)[1].strip() if ',' in name else name
    if '. ' in part:
        part = part.split('. ', 1)[1]
    return part.split()[0].strip('"\'.,')

df['FirstName'] = df.apply(lambda row: extract_first_name(row['Name'], row['Sex']), axis=1)
df[['Name', 'Sex', 'FirstName']].head()


## Самые популярные имена
Считаем отдельно для мужчин и женщин.

In [ ]:
def top_names(data: pd.DataFrame) -> pd.Series:
    counts = data['FirstName'].value_counts()
    return counts[counts == counts.max()]

print('Самые популярные мужские имена:')
display(top_names(df[df['Sex'] == 'male']))
print('Самые популярные женские имена:')
display(top_names(df[df['Sex'] == 'female']))


## Популярные имена по классам
Смотрим лидеров по имени для каждого пола внутри каждого класса.

In [ ]:
rows = []

for pclass in sorted(df['Pclass'].dropna().unique()):
    for sex in ['male', 'female']:
        subset = df[(df['Pclass'] == pclass) & (df['Sex'] == sex)]
        counts = subset['FirstName'].value_counts()
        top = counts[counts == counts.max()]
        rows.append({
            'Pclass': pclass,
            'Sex': sex,
            'TopNames': ', '.join(top.index.tolist()),
            'Count': int(top.iloc[0])
        })

pd.DataFrame(rows)


## Пассажиры старше 44 лет
Выводим часть таблицы по условию.

In [ ]:
df[df['Age'] > 44].head(10)


## Мужчины младше 44 лет
Выводим часть таблицы по двум условиям.

In [ ]:
df[(df['Age'] < 44) & (df['Sex'] == 'male')].head(10)


## Вместимость кабин
Считаем, сколько кают были 2-местными, 3-местными и далее.

In [ ]:
cabin_counts = df['Cabin'].dropna().str.split().explode().value_counts()
multi_seat_cabins = cabin_counts.value_counts().sort_index()
multi_seat_cabins = multi_seat_cabins[multi_seat_cabins.index > 1]
multi_seat_cabins.rename_axis('PlacesInCabin').reset_index(name='CabinsCount')


## Пассажиры без родственников
Считаем тех, у кого `SibSp + Parch = 0`.

In [ ]:
((df['SibSp'] + df['Parch']) == 0).sum()
